In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_parquet("df_integrated_dashboard.parquet")

In [14]:
df['is_rewarded'].value_counts()

is_rewarded
0    15361123
1     1469931
Name: count, dtype: int64

In [53]:
# --------------------------------
# Fraud 유저 정의
# --------------------------------

df["Abuser"] = np.where(
    (df["is_rewarded"] == 1) &
    (df["row_label"] == "극단값"),
    "Y",
    "N"
)

# --------------------------------
# Fraud 손실
# --------------------------------

df["fraud_loss"] = np.where(
    df["Abuser"] == "Y",
    df["adv_cost"] - df["media_price"],
    0
)

# --------------------------------
# CVR 계산용
# --------------------------------

df["reward_flag"] = df["is_rewarded"].astype(int)
df["is_fraud"] = (df["row_label"] == "극단값").astype(int)
df["fraud_reward"] = (df["adv_cost"] - df["earn_cost"])* (df["is_fraud"])

media_df = (
    df.groupby("mda_idx")
    .agg(

        clicks=("click_key","count"),

        rewards=("reward_flag","sum"),

        adv_cost=("adv_cost","sum"),

        earn_cost=("earn_cost","sum"),

        Risk_Label =("Risk_Label","first"),
        Risk_Label_score =("Risk_Label_score","first"),
        conversions=("is_rewarded","sum"),
        fraud_clicks=("is_fraud","sum"),
        fraud_loss=("fraud_reward","sum")
        
    )
    .reset_index()
)

media_df["CVR"] = (
    media_df["rewards"] /
    media_df["clicks"]
)
media_df["fraud_ratio"] = (
    media_df["fraud_clicks"] /
    media_df["clicks"]
)

media_df.to_parquet(
    "media_dashboard3.parquet",
    index=False
)

In [44]:
media_df.head()

,mda_idx,clicks,rewards,cost,revenue,Risk_Label,Risk_Label_score,conversions,fraud_clicks,fraud_loss,Total_Score,ctit_grade,is_rewarded,CVR,fraud_ratio
0,12,232385,82929,16169408.0,13168440.0,경고,정상,82929,4151,86900.0,79.464179,미적립,0,0.356860,0.017863
1,14,83094,22672,9561157.0,6889810.0,경고,정상,22672,4038,127936.0,59.844892,정상,1,0.272848,0.048596
2,18,4937,92,527416.0,392450.0,정상,정상,92,0,0.0,99.937194,정상,1,0.018635,0.000000
3,22,83355,35232,8605200.0,6063430.0,경고,정상,35232,2705,215048.0,79.464179,미적립,0,0.422674,0.032452
4,26,678,70,17170.0,14480.0,정상,정상,70,0,0.0,79.464179,미적립,0,0.103245,0.000000


In [45]:
# -------------------------------
# 전체 클릭 / 전환
# -------------------------------

total_click = df["click_key"].count()
total_conv = df["is_rewarded"].sum()

total_cvr = total_conv / total_click


# -------------------------------
# 극단값 제거 (정상 데이터)
# -------------------------------

clean_df = df[df["row_label"] != "극단값"]

clean_click = clean_df["click_key"].count()
clean_conv = clean_df["is_rewarded"].sum()

clean_cvr = clean_conv / clean_click


# -------------------------------
# CVR 변화
# -------------------------------

cvr_change = clean_cvr - total_cvr


# -------------------------------
# Fraud 전환
# -------------------------------

fraud_df = df[df["row_label"] == "극단값"]

fraud_click = fraud_df["click_key"].count()
fraud_conv = fraud_df["is_rewarded"].sum()

fraud_cvr = fraud_conv / fraud_click if fraud_click > 0 else 0


# -------------------------------
# Fraud 손실
# -------------------------------

fraud_loss = fraud_df["adv_cost"].sum()- fraud_df["earn_cost"].sum()


# -------------------------------
# KPI dataframe 생성
# -------------------------------

kpi_dashboard = pd.DataFrame({

    "total_click":[total_click],
    "total_conv":[total_conv],
    "total_cvr":[total_cvr],

    "clean_click":[clean_click],
    "clean_conv":[clean_conv],
    "clean_cvr":[clean_cvr],

    "fraud_click":[fraud_click],
    "fraud_conv":[fraud_conv],
    "fraud_cvr":[fraud_cvr],

    "fraud_loss":[fraud_loss],

    "cvr_change":[cvr_change]

})


# -------------------------------
# 저장
# -------------------------------

kpi_dashboard.to_parquet("kpi_dashboard.parquet")


In [38]:
kpi_dashboard.head()

,total_click,total_conv,total_cvr,clean_click,clean_conv,clean_cvr,fraud_click,fraud_conv,fraud_cvr,fraud_loss,cvr_change
0,16831054,1469931,0.087334,10443156,939144,0.089929,6387898,530787,0.083093,30058868116,0.002595


In [51]:
df.groupby("row_label").agg(
    clicks=("click_key","count"),
    conversions=("is_rewarded","sum"),
    cvr=("is_rewarded","mean")
)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


,clicks,conversions,cvr
row_label,,,
극단값,6387898,530787,0.083093
미적립,1849507,0,0.000000
어뷰징,7782516,128011,0.016449
정상,811133,811133,1.000000


시간대별 분석

In [57]:
# datetime 변환
df["click_date"] = pd.to_datetime(df["click_date"])

# hour 생성
df["hour"] = df["click_date"].dt.hour

# 정상 / 어뷰징 구분
df["type"] = df["row_label"].apply(
    lambda x: "어뷰징" if x == "극단값" else "정상"
)

# 시간대별 클릭
hour_df = (
    df.groupby(["hour","type"])
    .size()
    .reset_index(name="clicks")
)

# 타입별 전체 클릭
total = (
    df.groupby("type")
    .size()
    .reset_index(name="total_clicks")
)

# merge
hour_df = hour_df.merge(total, on="type")

# 비율
hour_df["ratio"] = hour_df["clicks"] / hour_df["total_clicks"]

# 저장
hour_df.to_parquet("hour_dashboard.parquet", index=False)

In [58]:
# 시간 생성
df["hour"] = pd.to_datetime(df["click_date"]).dt.hour

# 어뷰징 구분
df["type"] = "정상"
df.loc[df["row_label"] == "극단값", "type"] = "어뷰징"

# 매체 + 시간 + 타입 기준 클릭수
media_hour_df = (
    df.groupby(["mda_idx","hour","type"])
    .size()
    .reset_index(name="clicks")
)

# 저장
media_hour_df.to_parquet("hour_media_dashboard.parquet", index=False)

In [8]:
df["is_rewarded"].value_counts().sort_index()

is_rewarded
0    15361123
1     1469931
Name: count, dtype: int64

In [29]:
df_aviewser = df[(df["is_rewarded"] == 1) & (df['row_label']== "극단값")]

In [ ]:
df_aviewser.

,click_key,ads_idx,mda_idx,pub_sub_rel_id,user_ip,dvc_idx,click_date,click_time,reward_price,media_price,...,click_score,row_score,Total_Score,Risk_Label_score,Risk_Label,extreme_ratio,abusing_ratio,normal_ratio,unrewarded_ratio,click_hour
1,000002b4d92f7648b455877c2676452efcd22a09,412426,58,46032732,35.78.117.76,34422806,2025-07-26 02:18:24,2,170,170,...,40,61.0,59.844892,경고,매우위험,0.850827,0.143865,0.000141,0.005168,0
22,00001a0ece505b33b634a83a77a3600c2f87ff53,430804,56,144677810,13.231.138.185,52536734,2025-08-01 00:20:12,0,230,230,...,70,67.0,77.512897,경고,위험,0.266504,0.653799,0.014953,0.064744,0
70,00005076d9976099d2b4512a8eb4f8570a076084,441932,563,1,175.126.208.146,53749185,2025-08-11 09:15:37,9,14,14,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0
113,0000811631432c69235fc215986b83a50dd7b03d,435428,563,1,175.126.208.146,55407521,2025-07-30 06:29:06,6,14,14,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0
133,000096fd76b6e4b1f688625512cfef26e5d505b0,438831,563,1,175.126.208.146,55085197,2025-08-04 00:51:50,0,130,130,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16830957,8f697cdf397d282556f9742cd72c28103db4546f,437027,563,1,175.126.208.146,57136480,2025-07-27 07:08:18,7,14,14,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0
16830963,8f6984384a1202a338b23399e9728e30526a8c83,442664,563,1,175.126.208.146,55637719,2025-08-09 07:52:08,7,14,14,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0
16830993,8f69a5647adb15a2a9a598dfba7e2f1f97b26f9e,441776,563,1,175.126.208.146,44249561,2025-08-08 07:13:37,7,14,14,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0
16831018,8f69bafdbf6363fb66c7a30c276c100a1e1ed04b,444752,563,1,175.126.208.146,58471384,2025-08-18 08:42:54,8,14,14,...,10,55.0,54.415948,경고,매우위험,1.000000,0.000000,0.000000,0.000000,0


In [34]:
import numpy as np
df["Abuser"] = np.where(
    (df["is_rewarded"] == 1) & (df["row_label"] == "극단값"),
    "Y","N")

In [ ]:
loss_df = df[df["Abuser"] == "Y"]
df["fraud_loss"] = df["adv_cost"] - df["earn_cost"]

진짜 전환 가짜 전환 비교 

In [37]:
media_detail = (
    df.groupby("mda_idx")
    .agg(
        total_click=("click_key","count"),
        total_conv=("is_rewarded","sum"),

        fraud_click=("row_label", lambda x: (x=="극단값").sum()),

        fraud_conv=("is_rewarded",
            lambda x: (
                (df.loc[x.index,"row_label"]=="극단값") &
                (x==1)
            ).sum()
        )
    )
    .reset_index()
)
# 정상 계산
media_detail["normal_click"] = media_detail["total_click"] - media_detail["fraud_click"]
media_detail["normal_conv"] = media_detail["total_conv"] - media_detail["fraud_conv"]

# 비율 계산
media_detail["fraud_ratio"] = media_detail["fraud_click"] / media_detail["total_click"]

media_detail["fraud_cvr"] = (
    media_detail["fraud_conv"] /
    media_detail["fraud_click"].replace(0,1)
)

media_detail["normal_cvr"] = (
    media_detail["normal_conv"] /
    media_detail["normal_click"].replace(0,1)
)
# dashboard용 컬럼
media_detail_dashboard = media_detail[
    [
        "mda_idx",
        "normal_click",
        "normal_conv",
        "fraud_click",
        "fraud_conv",
        "fraud_ratio",
        "fraud_cvr",
        "normal_cvr"
    ]
]

media_detail_dashboard.to_parquet("media_detail_dashboard.parquet")

In [42]:
media_detail_dashboard.head()

,mda_idx,normal_click,normal_conv,fraud_click,fraud_conv,fraud_ratio,fraud_cvr,normal_cvr
0,12,228234,81388,4151,1541,0.017863,0.371236,0.356599
1,14,79056,20456,4038,2216,0.048596,0.548787,0.258753
2,18,4937,92,0,0,0.000000,0.000000,0.018635
3,22,80650,33933,2705,1299,0.032452,0.480222,0.420744
4,26,678,70,0,0,0.000000,0.000000,0.103245
